# Sentiment Analysis - Resturant Reviews Classification

### Data Importing


In [22]:
import numpy as np
import pandas as pd
import re


In [23]:
df = pd.read_csv(r"a1_RestaurantReviews_HistoricDump.tsv", sep='\t', quoting = 3)
df.shape

(900, 2)

In [24]:
df.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [25]:
df['Liked'].value_counts()

Liked
1    496
0    404
Name: count, dtype: int64

### Data Preprocessing

In [26]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

all_stopwords = stopwords.words('english')
for word in ('not', 'no', 'nor', 'never'):
    if word in all_stopwords:
        all_stopwords.remove(word)  # keep negation words to preserve sentiment
all_stopwords = set(all_stopwords)  # set lookup is faster

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\masel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [27]:
def preprocess(review):
    review = re.sub('[^a-zA-Z]', ' ', review) #Remove all characters except letters
    review = review.lower().split()  #Convert all letters to lowercase & Split the review into individual words
    review = [ps.stem(word) for word in review if not word in all_stopwords] #Remove stopwords and apply stemming to the remaining words
    return ' '.join(review) #Join the processed words back into a single string

corpus = [preprocess(df['Review'][i]) for i in range(df.shape[0])]

y = df['Liked'].values #Extract the target variable (Liked) as a numpy array

corpus[:10]

['wow love place',
 'crust not good',
 'not tasti textur nasti',
 'stop late may bank holiday rick steve recommend love',
 'select menu great price',
 'get angri want damn pho',
 'honeslti tast fresh',
 'potato like rubber could tell made ahead time kept warmer',
 'fri great',
 'great touch']

### Data Transformation

Train-Test Split

In [28]:
from sklearn.model_selection import train_test_split
X = corpus
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0, stratify = y) #Split the dataset into training and testing sets (80% training, 20% testing)

print('Training reviews:', len(X_train))
print('Test reviews:    ', len(X_test))

Training reviews: 720
Test reviews:     180


In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1420, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=0))
])

param_grid = {
    'tfidf__max_features': [500, 1000, 1500, 2000, 3000, None],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy: {:.2f}%".format(grid_search.best_score_ * 100))

Best parameters: {'clf__C': 10, 'tfidf__max_features': 1500, 'tfidf__ngram_range': (1, 1)}
Best cross-validation accuracy: 79.86%


In [30]:
results = pd.DataFrame(grid_search.cv_results_)
mask = (
    (results['param_clf__C'] == grid_search.best_params_['clf__C']) &
    (results['param_tfidf__ngram_range'] == grid_search.best_params_['tfidf__ngram_range'])
)
sweep = results[mask][['param_tfidf__max_features',
                       'mean_test_score', 'std_test_score']]
sweep = sweep.sort_values('mean_test_score', ascending=False)
sweep.rename(columns={
    'param_tfidf__max_features': 'max_features',
    'mean_test_score': 'mean_cv_accuracy',
    'std_test_score': 'std_cv_accuracy'})

,max_features,mean_cv_accuracy,std_cv_accuracy
42,2000,0.798611,0.023652
40,1500,0.798611,0.023652
44,3000,0.798611,0.023652
46,None,0.798611,0.023652
38,1000,0.794444,0.019934
36,500,0.784722,0.034021


### Model Fitting


In [31]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

best_model = grid_search.best_estimator_ #Get the best model from the grid search
y_pred = best_model.predict(X_test) #Predict the labels for the test set using the best model

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred)) #Print a classification report showing precision, recall, and F1-score for each class

Confusion Matrix:
[[60 21]
 [15 84]]

Accuracy: 0.8

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.74      0.77        81
           1       0.80      0.85      0.82        99

    accuracy                           0.80       180
   macro avg       0.80      0.79      0.80       180
weighted avg       0.80      0.80      0.80       180



### Model Performance

In [32]:
import joblib

joblib.dump(best_model, r"sentiment_analysis_model.pkl") #Save the best model to a file for later use (Sentiment Analysis model)
print("Model saved as 'sentiment_analysis_model.pkl'")

Model saved as 'sentiment_analysis_model.pkl'


## Predicting on the Fresh Dataset

In [33]:
fresh = pd.read_csv(r"a2_RestaurantReviews_FreshDump.tsv", sep='\t', quoting=3)
fresh_corpus = [preprocess(t) for t in fresh['Review']]

model = joblib.load(r"sentiment_analysis_model.pkl")
fresh['Predicted_Liked'] = model.predict(fresh_corpus)

print('Predicted positive:', int((fresh['Predicted_Liked'] == 1).sum()))
print('Predicted negative:', int((fresh['Predicted_Liked'] == 0).sum()))
fresh.head(10)

Predicted positive: 34
Predicted negative: 66


,Review,Predicted_Liked
0,Spend your money elsewhere.,0
1,Their regular toasted bread was equally satisf...,1
2,The Buffet at Bellagio was far from what I ant...,1
3,"And the drinks are WEAK, people!",1
4,-My order was not correct.,0
5,"Also, I feel like the chips are bought, not ma...",0
6,After the disappointing dinner we went elsewhe...,1
7,The chips and sals a here is amazing!!!!!!!!!!...,1
8,We won't be returning.,0
9,This is my new fav Vegas buffet spot.,1
